# Traffic Demand Prediction - Optimized Approach

This notebook implements key improvements over previous approaches to maximize the evaluation score:
1. **Geohash Decoding:** Converts geohash strings into continuous `latitude` and `longitude` to let tree models understand spatial proximity.
2. **Historical Target Encoding:** Feeds the model the historical average demand of a location to set a strong baseline.
3. **Time-based Validation Split:** Prevents data leakage by training on past data (`day == 48`) and validating on future data (`day == 49`).
4. **Cyclical Time Features:** Uses sine/cosine transformations on minutes to model the continuous nature of a 24-hour cycle.
5. **Hyperparameter Tuning & Early Stopping:** Drastically reduces overfitting by lowering `max_depth`, `learning_rate`, and employing `early_stopping_rounds`.

In [ ]:
# Install required dependencies
!pip install pygeohash xgboost

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import pygeohash as pgh
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings("ignore")

### 1. Load Datasets

In [ ]:
train_url = "https://raw.githubusercontent.com/atharv-d21/traffic_demand_prediction/refs/heads/main/datasets/train.csv"
test_url = "https://raw.githubusercontent.com/atharv-d21/traffic_demand_prediction/refs/heads/main/datasets/test.csv"

train = pd.read_csv(train_url)
test = pd.read_csv(test_url)

print(f"Train shape: {train.shape}, Test shape: {test.shape}")

### 2. Basic Preprocessing & Missing Values

In [ ]:
def preprocess_basics(df):
    df = df.copy()
    
    # Fill categorical nulls
    df["RoadType"] = df["RoadType"].fillna("Unknown")
    df["Weather"] = df["Weather"].fillna("Unknown")

    # Map binary features
    df["LargeVehicles"] = df["LargeVehicles"].replace({"Allowed": 1, "Not Allowed": 0}).fillna(0)
    df["Landmarks"] = df["Landmarks"].replace({"Yes": 1, "No": 0}).fillna(0)
    
    # Impute Temperature: Use mean per Weather condition, fallback to global mean
    df["Temperature"] = df.groupby("Weather")["Temperature"].transform(lambda x: x.fillna(x.mean()))
    df["Temperature"] = df["Temperature"].fillna(df["Temperature"].mean())
    
    return df

train = preprocess_basics(train)
test = preprocess_basics(test)

### 3. Geohash Decoding (Spatial Features)

In [ ]:
# Decode geohash into numeric latitude and longitude
train["latitude"] = train["geohash"].apply(lambda x: pgh.decode(x)[0])
train["longitude"] = train["geohash"].apply(lambda x: pgh.decode(x)[1])

test["latitude"] = test["geohash"].apply(lambda x: pgh.decode(x)[0])
test["longitude"] = test["geohash"].apply(lambda x: pgh.decode(x)[1])

### 4. Advanced Time Features (Cyclical)

In [ ]:
def add_time_features(df):
    df = df.copy()
    
    # Extract hour and minute
    df[["hour", "minute"]] = df["timestamp"].str.split(":", expand=True).astype(int)
    
    # Continuous time in minutes
    df["time_in_mins"] = df["hour"] * 60 + df["minute"]
    
    # Cyclical representations (sine and cosine)
    df["sin_time"] = np.sin(2 * np.pi * df["time_in_mins"] / 1440)
    df["cos_time"] = np.cos(2 * np.pi * df["time_in_mins"] / 1440)
    
    # Part of day categorization
    def get_part_of_day(hour):
        if 5 <= hour < 12: return "Morning"
        elif 12 <= hour < 17: return "Afternoon"
        elif 17 <= hour < 21: return "Evening"
        else: return "Night"
    
    df["part_of_day"] = df["hour"].apply(get_part_of_day)
    
    # Drop original timestamp
    df.drop(["timestamp"], axis=1, inplace=True)
    return df

train = add_time_features(train)
test = add_time_features(test)

### 5. Historical Target Encoding

In [ ]:
# To avoid data leakage, we calculate the historical average demand strictly from Day 48
historical_avg = train[train["day"] == 48].groupby("geohash")["demand"].mean().reset_index()
historical_avg.rename(columns={"demand": "hist_avg_demand"}, inplace=True)

# Merge back into both train and test sets
train = train.merge(historical_avg, on="geohash", how="left")
test = test.merge(historical_avg, on="geohash", how="left")

# Fill NaNs (locations in Day 49 or Test that were not seen in Day 48) with global mean of Day 48
global_mean_day48 = train[train["day"] == 48]["demand"].mean()
train["hist_avg_demand"] = train["hist_avg_demand"].fillna(global_mean_day48)
test["hist_avg_demand"] = test["hist_avg_demand"].fillna(global_mean_day48)

# Drop the geohash string column now that we have lat/lon and historical averages
train.drop(["geohash"], axis=1, inplace=True)
test.drop(["geohash"], axis=1, inplace=True)

### 6. Encoding Categorical Columns

In [ ]:
categorical_cols = ["RoadType", "Weather", "part_of_day"]
train = pd.get_dummies(train, columns=categorical_cols)
test = pd.get_dummies(test, columns=categorical_cols)

# Align test columns to train columns in case of mismatched categories
test = test.reindex(columns=[col for col in train.columns if col != "demand"], fill_value=0)

### 7. Time-based Splitting (Crucial for Time-Series validation)

In [ ]:
# Separate features and target
features = train.drop(["demand", "Index"], axis=1)
target = train["demand"]

# Train on Day 48
X_train = features[train["day"] == 48]
y_train = target[train["day"] == 48]

# Validate on Day 49 (matches Test data nature)
X_val = features[train["day"] == 49]
y_val = target[train["day"] == 49]

print(f"Training set: {X_train.shape}, Validation set: {X_val.shape}")

### 8. Model Training with Early Stopping

In [ ]:
# Initialize XGBoost with conservative hyperparameters to prevent overfitting
xgb_model = xgb.XGBRegressor(
    n_estimators=1500,       # High number of trees
    max_depth=6,             # Kept low to prevent memorizing sparse data
    learning_rate=0.03,      # Slower learning rate for stability
    subsample=0.8,           # Use 80% of rows per tree
    colsample_bytree=0.8,    # Use 80% of columns per tree
    random_state=42,
    objective="reg:squarederror"
)

# Train with early stopping on the validation set
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=100
)

### 9. Validation Evaluation & Feature Importance

In [ ]:
y_pred_val = xgb_model.predict(X_val)
val_r2 = r2_score(y_val, y_pred_val)
val_mse = mean_squared_error(y_val, y_pred_val)

print(f"Validation MSE: {val_mse:.5f}")
print(f"Validation R2 Score: {val_r2:.5f}")
print(f"\nEstimated Final Evaluation Score: {max(0, 100 * val_r2):.2f}")

# Plot Feature Importances
plt.figure(figsize=(10, 6))
xgb.plot_importance(xgb_model, max_num_features=15, importance_type="weight")
plt.title("Top 15 Feature Importances")
plt.show()

### 10. Generate Predictions on Test Set

In [ ]:
test_features = test.drop(["Index"], axis=1)

test_predictions = xgb_model.predict(test_features)

submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": test_predictions
})

submission.to_csv("submissions/submission_optimized_xgb.csv", index=False)

print("Submission file successfully created!")
display(submission.head())